# Notebook 1 — Phân tích dữ liệu khám phá (EDA) và tiền xử lý

Notebook này giới thiệu dữ liệu **VialectBench**.
Ta không bắt đầu bằng mô hình; ta bắt đầu bằng schema, split và câu hỏi nghiên cứu.
Mục tiêu của notebook này là giúp nhóm hiểu rõ dữ liệu trước khi chạy bất kỳ
thí nghiệm nào với mô hình ngôn ngữ.

## Bối cảnh dự án

**Câu hỏi nghiên cứu trung tâm:**

> Một mô hình có giữ nguyên hành vi khi câu tiếng Việt chuẩn được viết lại bằng
> phương ngữ nhưng không đổi nghĩa không, và text normalization có giảm khoảng
> cách đó không?

Ba phương ngữ được dùng trong đồ án:

- `PNB` — phương ngữ Bắc (Bắc Bộ).
- `PNT2` — Bắc Trung Bộ 2 (Nghệ An - Hà Tĩnh).
- `PNT3` — Bắc Trung Bộ 3 (Quảng Bình - Quảng Trị - Thừa Thiên Huế).

Bốn task gốc: `MCQA`, `NLI`, `QA`, `SENT`.

## Mục tiêu học tập

Sau notebook này, nhóm có thể:

1. Kiểm tra cân bằng dữ liệu và phát hiện leakage theo `sample_id`.
2. Giải thích vì sao NLI phải chuẩn hóa **hypothesis** chứ không phải premise.
3. Đo khác biệt bề mặt giữa câu chuẩn và câu phương ngữ (độ dài, character edit rate).
4. Đọc kết quả model degradation đã có sẵn theo model, task và dialect.
5. Viết **insight có bằng chứng** thay vì chỉ mô tả biểu đồ.

## Cách dùng notebook này

Notebook chia thành hai loại cell:

- **Code sẵn (instructive):** pipeline tối thiểu để học viên dùng chung dữ liệu
  và metric. Nhóm đọc, chạy và hiểu — không cần sửa.
- **Bài tập của nhóm:** đánh dấu bằng `STUDENT TASK`, có `HINT` và marker
  `Your code here` (viết trong ba ngoặc kép). Nhóm thay marker bằng code của
  mình, chạy cell và vượt qua `SELF-CHECK`.

Nguyên tắc nghiên cứu: **không sửa test split, không nhìn test để chọn thiết kế.**
Nhóm có thể thay biểu đồ/mô hình/hyperparameter nếu ghi rõ lý do và giữ một
baseline so sánh công bằng.

In [ ]:
# STUDENT TASK 0 — Đăng ký nhóm và câu hỏi nghiên cứu.
# Đây là "preregistration": ghi lại dự đoán TRƯỚC khi xem kết quả.
# HINT: RQ nên nêu rõ (1) input/population, (2) yếu tố so sánh, (3) metric.
"""Your code here"""
TEAM_NAME = "TODO"
RESEARCH_QUESTION = "TODO: ví dụ - PNT3 có gây degradation lớn hơn PNB không?"
HYPOTHESIS = "TODO: dự đoán có hướng, ví dụ 'PNT3 > PNT2 > PNB về delta-NLL'"
PRIMARY_METRIC = "TODO: ví dụ delta-NLL hoặc CER"

print({
    "team": TEAM_NAME,
    "rq": RESEARCH_QUESTION,
    "hypothesis": HYPOTHESIS,
    "primary_metric": PRIMARY_METRIC,
})

## Nạp dữ liệu và schema

Hai file chính:

- `data/train_240.jsonl` — 240 cặp (20 source/task × 4 task × 3 dialect).
- `data/test_300.jsonl` — 300 cặp giữ lại (25 source/task × 4 task × 3 dialect).

Mỗi dòng là một cặp `(standard_text, dialect_text)`. Với cùng `sample_id`,
ba dòng tương ứng PNB, PNT2 và PNT3. Vì vậy train/test phải tách theo
**source (`sample_id`)**, không tách ngẫu nhiên từng dòng.

In [ ]:
from pathlib import Path
import sys

# Detect project root whether we run from notebooks/ or project root.
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "data" / "train_240.jsonl").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run the notebook from the project root or notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))
print(f"Project root: {ROOT}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from vialect_seas.data import DIALECTS, TASKS, identity_rate_summary, load_jsonl

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 100)

train = load_jsonl(ROOT / "data" / "train_240.jsonl")
test = load_jsonl(ROOT / "data" / "test_300.jsonl")
print(f"train={len(train):,}  test={len(test):,}")
print(f"dialects: {DIALECTS}")
print(f"tasks: {TASKS}")
train.head(3)

### STUDENT TASK 1 — Đọc schema bằng ví dụ

Chọn **cùng một `sample_id`** và trình bày ba biến thể PNB, PNT2, PNT3 cạnh nhau.
Sau đó ghi một nhận xét ngắn về: phần **nội dung** được giữ nguyên và phần
**ngôn ngữ** đã thay đổi.

In [ ]:
RUN_STUDENT_TASK_1 = False  # Đổi thành True sau khi viết xong code.

if RUN_STUDENT_TASK_1:
    # HINT 1: train.groupby("sample_id").size() cho biết source nào có đủ 3 dialect.
    # HINT 2: Chọn các cột sample_id, task, target_dialect, standard_text, dialect_text.
    """Your code here"""

    # SELF-CHECK: gán kết quả vào biến `student_examples`.
    assert "student_examples" in globals(), "Hãy tạo biến student_examples"
    assert student_examples["sample_id"].nunique() == 1
    assert set(student_examples["target_dialect"]) == {"PNB", "PNT2", "PNT3"}
    display(student_examples)

### Insight

Dữ liệu đã ở dạng **paired normalization**: input là `dialect_text`, reference
là `standard_text`. Các trường `label`/`task` vẫn được giữ để kiểm tra mô hình
có phá cấu trúc hoặc semantics của task gốc hay không.

**Nhóm bổ sung:** Chọn hai dòng có cấu trúc khác nhau (ví dụ một câu SENT ngắn
và một câu MCQA dài) và giải thích mô hình normalization sẽ phải học phép biến
đổi gì. Ghi vào báo cáo.

## Quy tắc đặc biệt cho NLI

Trong benchmark này, task NLI có cấu trúc:

- `source_text` = **premise** (đoạn văn dài).
- `standard_text` = **hypothesis** chuẩn (câu ngắn).
- `dialect_text` = hypothesis đã viết lại bằng phương ngữ.

**Điều này rất quan trọng:** nếu nhóm dùng `source_text` (premise) làm target
normalization, mô hình sẽ học một phép biến đổi sai (premise → dialect) và kết
quả sẽ vô nghĩa. Luôn chuẩn hóa **hypothesis**.

In [ ]:
from vialect_seas.data import assert_balanced_split

# Kiểm tra cấu trúc dữ liệu: cân bằng + không leakage + NLI đúng target.
assert_balanced_split(train, sources_per_task=20)
assert_balanced_split(test, sources_per_task=25)

overlap = set(train["sample_id"]) & set(test["sample_id"])
nli_target_ok = (
    train.query("task == 'NLI'")["standard_text"]
    == train.query("task == 'NLI'")["hypothesis"]
).all()

print("Train/test source overlap:", len(overlap), "(phải = 0)")
print("NLI standard_text == hypothesis:", nli_target_ok, "(phải = True)")
print("Missing values:\n", train[["dialect_text", "standard_text"]].isna().sum())

### Insight

- **`0` source overlap** là điều kiện để đánh giá generalization. Nếu cùng source
  xuất hiện ở train (dưới PNT2) và test (dưới PNT3), mô hình có thể nhớ nội dung
  chuẩn thay vì học normalization.
- **NLI target = hypothesis** đảm bảo mô hình học đúng phép biến đổi
  (dialect hypothesis → standard hypothesis), không phải premise → dialect.

**Nhóm bổ sung:** Mô tả một ví dụ leakage cụ thể (giả thuyết) bằng `sample_id`.

## Quality control — typo và identity pairs

Identity pair (`dialect_text == standard_text`) không mặc nhiên là lỗi: một câu
có thể không cần thay đổi. Tuy nhiên tỷ lệ copy cao có thể làm CER trung bình
trông tốt hơn dù normalizer chưa học chuyển đổi. Vì vậy ta báo cáo tỷ lệ này theo
task và dialect, rồi audit thủ công theo `sample_id`.

In [ ]:
identity_summary = identity_rate_summary(train)
display(identity_summary.round(3))

suspicious_fragments = ["thông inh"]
suspicious_rows = train[
    train["standard_text"].str.contains(
        "|".join(suspicious_fragments), case=False, na=False
    )
]
print("Known suspicious target rows:", len(suspicious_rows))
assert suspicious_rows.empty, "Dữ liệu còn typo target đã biết; chạy prepare_student_data.py"

### Insight (quality control)

Báo cáo identity rate theo dialect và giải thích vì sao PNB có thể có nhiều cặp
copy hơn PNT2/PNT3. Chọn ngẫu nhiên 20–30 `sample_id` để audit Unicode, khoảng
trắng, dấu câu, typo target và bảo toàn nghĩa. Không xóa identity pair chỉ vì nó
giống reference.

## EDA 1 — Cân bằng dữ liệu

Đếm số cặp theo task × dialect. Thiết kế cân bằng giúp average không bị một
dialect/task nhiều mẫu hơn chi phối.

In [ ]:
composition = (
    train.groupby(["task", "target_dialect"], observed=True)
    .size().rename("rows").reset_index()
)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=composition, x="task", y="rows", hue="target_dialect", ax=ax)
ax.set(title="Số cặp theo task và dialect (train)", xlabel="Task",
       ylabel="Số cặp", ylim=(0, 24))
ax.legend(title="Dialect", ncol=3)
plt.show()
display(composition)

### Insight

Mỗi ô task-dialect có đúng 20 cặp. Thiết kế cân bằng giúp so sánh công bằng
giữa các dialect, nhưng **không phản ánh tần suất dialect trong đời thực**
(đây là pilot study, không phải corpus tự nhiên).

**Nhóm bổ sung:** Nêu một ưu điểm và một hạn chế khác của balanced design.

## EDA 2 — Khác biệt bề mặt: độ dài và character edit rate

Hai feature đơn giản:

- **Length ratio** = số từ dialect / số từ standard. Gần 1 nghĩa là rewrite
  không dài/ngắn bất thường.
- **Character Error Rate (CER)** = khoảng cách edit (Levenshtein) trên số ký tự
  reference. Đo mức thay đổi bề mặt.

```text
CER = (substitutions + deletions + insertions) / |reference characters|
```

CER thấp hơn = giống chuẩn hơn. Nhưng CER không phải metric semantic: hai câu
khác nghĩa hoàn toàn vẫn có thể có CER thấp.

In [ ]:
from vialect_seas.metrics import character_error_rate

def add_surface_features(frame):
    result = frame.copy()
    result["standard_words"] = result["standard_text"].str.split().str.len()
    result["dialect_words"] = result["dialect_text"].str.split().str.len()
    result["length_ratio"] = result["dialect_words"] / result["standard_words"].replace(0, np.nan)
    result["char_error"] = [
        character_error_rate(ref, dia)
        for ref, dia in zip(result["standard_text"], result["dialect_text"])
    ]
    return result

train_features = add_surface_features(train)
surface_summary = (
    train_features.groupby(["task", "target_dialect"], observed=True)
    .agg(mean_length_ratio=("length_ratio", "mean"),
         mean_char_error=("char_error", "mean"))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=train_features, x="target_dialect", y="length_ratio",
            hue="task", ax=axes[0])
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[0].set(title="Tỷ lệ độ dài dialect/standard", xlabel="Dialect",
            ylabel="Tỷ lệ số từ")
sns.barplot(data=surface_summary, x="target_dialect", y="mean_char_error",
            hue="task", ax=axes[1])
axes[1].set(title="Character edit rate trung bình", xlabel="Dialect",
            ylabel="CER")
plt.tight_layout()
plt.show()
display(surface_summary.round(3))

### Insight

- **Length ratio ≈ 1** không chứng minh hai câu cùng nghĩa; nó chỉ loại trừ
  việc rewrite dài/ngắn bất thường.
- **CER** cho biết mức thay đổi bề mặt, nhưng không phải metric semantic. Vì
  vậy cần giữ `label`/`task` constraints và xem lỗi thủ công ở Notebook 3.
- Thường PNT2/PNT3 có CER cao hơn PNB — quan sát này gợi ý dialect Trung Bộ
  thay đổi bề mặt nhiều hơn.

**Nhóm bổ sung:** Dẫn hai giá trị cụ thể từ biểu đồ và giải thích chúng có/không
hỗ trợ hypothesis ban đầu.

## EDA 3 — Model performance và degradation có sẵn

Hai bảng sau được tính từ kết quả direct prompting của 10 mô hình (Qwen, Llama,
Mistral, Gemma, Vistral, SeaLLM, GPT-4o,...). `Drop = Standard - Dialect`;
số dương nghĩa là mô hình **giảm điểm** trên dialect.

In [ ]:
by_task = pd.read_csv(ROOT / "data" / "model_results" / "performance_by_model_task.csv")
task_matrix = pd.read_csv(ROOT / "data" / "model_results" / "task_performance_matrix_precise.csv")

model_overview = task_matrix[["Model", "All Standard", "All Dialect", "All Delta"]].copy()
model_overview[["All Standard", "All Dialect", "All Delta"]] *= 100
model_overview = model_overview.rename(columns={
    "All Standard": "Standard", "All Dialect": "Dialect", "All Delta": "Drop"
}).sort_values("Dialect", ascending=False).reset_index(drop=True)

plot_data = model_overview.melt(
    id_vars=["Model", "Drop"], value_vars=["Standard", "Dialect"],
    var_name="Variant", value_name="Score"
)
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=plot_data, x="Model", y="Score", hue="Variant", ax=ax)
ax.set(title="Standard vs. mean dialect performance", xlabel="", ylabel="Score (%)")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()
display(model_overview.round(2))

### Insight

**Absolute score và robustness không giống nhau.** Một mô hình mạnh (Standard
cao) vẫn có thể có `Drop > 0`; do đó báo cáo chỉ Standard score sẽ che khuất
sensitivity với dialect. Robustness phải đo bằng **paired drop**, không phải
absolute score.

**Nhóm bổ sung:** Chọn hai mô hình để minh họa bằng số (một mô hình mạnh nhưng
drop cao, một mô hình yếu nhưng drop thấp).

In [ ]:
task_dialect_matrix = pd.read_csv(
    ROOT / "data" / "model_results" / "task_dialect_performance_matrix_precise.csv"
)

# Tính paired drop cho từng (model, dialect) trên cả 6 dialect gốc.
model_columns = list(task_dialect_matrix.columns[2:])
drop_rows = []
for task in task_dialect_matrix["Task"].unique():
    task_rows = task_dialect_matrix[task_dialect_matrix["Task"] == task]
    standard_row = task_rows[task_rows.Variant == "Standard"].iloc[0]
    for dialect in ["PNB", "PNN", "PNT1", "PNT2", "PNT3", "PNT4"]:
        dialect_row = task_rows[task_rows.Variant == dialect].iloc[0]
        for model_name in model_columns:
            drop_rows.append({
                "Model": model_name,
                "Dialect": dialect,
                "Drop": 100 * (standard_row[model_name] - dialect_row[model_name]),
            })
precise_drops = pd.DataFrame(drop_rows)
drop_matrix = (
    precise_drops.groupby(["Model", "Dialect"], observed=True)
    .Drop.mean().unstack()
)
drop_matrix = drop_matrix[["PNB", "PNN", "PNT1", "PNT2", "PNT3", "PNT4"]]
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(drop_matrix, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax)
ax.set(title="Paired performance degradation (pp)", xlabel="Dialect", ylabel="Model")
plt.tight_layout()
plt.show()

# Chỉ 3 dialect của đồ án:
selected = drop_matrix[["PNB", "PNT2", "PNT3"]].mean().sort_values()
display(selected.rename("Mean drop across models (pp)").to_frame())

### Insight

Trong ba dialect của đồ án, **PNT3 và PNT2 có degradation lớn hơn PNB** trên
trung bình các mô hình. Đây là quan sát để hình thành giả thuyết, **chưa phải
giải thích nhân quả**: độ khác biệt từ vựng, tokenizer và dữ liệu pretraining
đều có thể góp phần.

**Nhóm bổ sung:** Viết một alternative explanation (ví dụ: tokenizer confound)
và một thí nghiệm có thể phân biệt hai giải thích.

## Bài tập mở

1. Viết một RQ và dự đoán thứ tự khó của ba dialect trước khi chạy Notebook 2.
2. Tìm ba cặp có CER cao nhất ở mỗi task; kiểm tra nghĩa có còn giữ không.
3. Thiết kế thêm một biểu đồ EDA và viết **Insight** gồm: observation, evidence,
   caveat.
4. Với MCQA, kiểm tra rewrite có giữ đủ lựa chọn (A/B/C/D) và đáp án đúng không.
5. Ghi kết quả EDA vào `outputs/team_eda_summary.csv`.

In [ ]:
# Lưu artifact nộp bài.
output_dir = ROOT / "outputs"
output_dir.mkdir(exist_ok=True)
surface_summary.to_csv(output_dir / "team_eda_summary.csv", index=False)
print("Saved", output_dir / "team_eda_summary.csv")

In [ ]:
# STUDENT TASK 2 (EXTENSION) — Thiết kế ít nhất một EDA mới.
# Cell không chạy cho tới khi nhóm bật cờ.
RUN_STUDENT_EDA = False

def build_student_eda(frame):
    # HINT 1: Chọn một feature chưa có (ví dụ: tỷ lệ token trùng, n-gram overlap).
    # HINT 2: Aggregate theo task/dialect; thêm uncertainty nếu phù hợp.
    # HINT 3: Trả về dict có hai khóa: "summary" (DataFrame) và "figure" (matplotlib Figure).
    """Your code here"""
    return None

if RUN_STUDENT_EDA:
    student_result = build_student_eda(train_features)
    if student_result is None:
        raise NotImplementedError("Hoàn thành build_student_eda trước khi bật RUN_STUDENT_EDA")
    # SELF-CHECK
    assert isinstance(student_result, dict)
    assert {"summary", "figure"}.issubset(student_result)
    display(student_result["summary"])
    plt.show()